# Notebook Name: 03_hypothesis_testing.ipynb

## Objectives
* Conduct hypothesis testing to evaluate relationships between key clinical variables and 30-day hospital readmissions (`readmitted_30`).
* Perform Chi-Square tests for categorical variables and Two-Sample T-Tests for continuous numerical variables.
* Measure effect sizes to confirm practical significance

## Inputs
* `data/processed/cleaned_diabetic_data.csv`

## Outputs
* Statistical test statistics, p-values, and clinical conclusions.

## The Pipeline
1. **Executive Overview**
2. **Environment Setup & Data Loading**
3. **Hypothesis Test 1: Prior Inpatient Visits vs Readmission**
4. **Hypothesis Test 2: Length of Stay vs Readmission**
5. **Hypothesis Test 3: Medication Changes vs Readmission**
6. **Preparation Summary**
7. **Version Control Workflow**
8. **Next Step**
9. **Acknowledgements and Credits**
10. **Sources and Layout Inspiration**
11. **Tooling and Assistance**

## Executive Overview

This notebook validates whether observed patterns in patient readmissions are statistically significant or attributable to random sampling variation. I tested three specific clinical hypotheses using Chi-Square tests of independence and Two-Sample T-Tests at a standard significance threshold of $\alpha = 0.05$. Effect sizes are calculated to quantify the magnitude of each relationship.

## Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load processed dataset
df = pd.read_csv('../data/processed/cleaned_diabetic_data.csv')

print(f"Dataset successfully loaded for hypothesis testing.")
print(f"Total Patient Records: {df.shape[0]}")

Dataset successfully loaded for hypothesis testing.
Total Patient Records: 69990


## Hypothesis Test 1: Prior Inpatient Visits vs Readmission

* **Null Hypothesis ($H_0$):** Prior inpatient admissions are independent of 30-day hospital readmissions.
* **Alternative Hypothesis ($H_1$):** Prior inpatient admissions are significantly associated with 30-day hospital readmissions.
* **Test Method:** Chi-Square ($\chi^2$) Test of Independence ($\alpha = 0.05$).

In [7]:
# Create contingency table (Has prior inpatient visit vs Readmitted within 30 days)
contingency_inpatient = pd.crosstab(df['number_inpatient'] > 0, df['readmitted_30'])

# Run Chi-Square test
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_inpatient)

# Calculate Cramér's V (Effect Size)
n = contingency_inpatient.sum().sum()
min_dim = min(contingency_inpatient.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print("=== CHI-SQUARE TEST: PRIOR INPATIENT VISITS ===")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"p-value: {p_val:.4e}")
print(f"Degrees of Freedom: {dof}")
print(f"Cramér's V (Effect Size): {cramers_v:.4f}\n")

if p_val < 0.05:
    print("Conclusion: Reject H0. Prior inpatient visits have a statistically significant association with 30-day readmission.")
else:
    print("Conclusion: Fail to reject H0.")

=== CHI-SQUARE TEST: PRIOR INPATIENT VISITS ===
Chi-Square Statistic: 467.1315
p-value: 1.3489e-103
Degrees of Freedom: 1
Cramér's V (Effect Size): 0.0817

Conclusion: Reject H0. Prior inpatient visits have a statistically significant association with 30-day readmission.


## Hypothesis Test 2: Length of Stay vs Readmission

* **Null Hypothesis ($H_0$):** Mean hospital stay duration (`time_in_hospital`) is equal for patients readmitted within 30 days and those not readmitted.
* **Alternative Hypothesis ($H_1$):** Mean hospital stay duration is significantly different for patients readmitted within 30 days.
* **Test Method:** Two-Sample Welch's T-Test ($\alpha = 0.05$).

In [8]:
# Separate stay duration by readmission status
stay_not_readmitted = df[df['readmitted_30'] == 0]['time_in_hospital']
stay_readmitted = df[df['readmitted_30'] == 1]['time_in_hospital']

# Perform Welch's T-Test (equal_var=False)
t_stat, p_val_t = stats.ttest_ind(stay_readmitted, stay_not_readmitted, equal_var=False)

# Calculate Cohen's d (Effect Size)
n1, n2 = len(stay_readmitted), len(stay_not_readmitted)
s1, s2 = stay_readmitted.std(), stay_not_readmitted.std()
s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
cohens_d = (stay_readmitted.mean() - stay_not_readmitted.mean()) / s_pooled

print("=== TWO-SAMPLE T-TEST: LENGTH OF STAY ===")
print(f"Mean Stay (Readmitted <30d): {stay_readmitted.mean():.2f} days")
print(f"Mean Stay (Not Readmitted):  {stay_not_readmitted.mean():.2f} days")
print(f"T-Statistic: {t_stat:.4f}")
print(f"p-value: {p_val_t:.4e}")
print(f"Cohen's d (Effect Size): {cohens_d:.4f}\n")

if p_val_t < 0.05:
    print("Conclusion: Reject H0. Hospital length of stay is significantly longer for patients readmitted within 30 days.")
else:
    print("Conclusion: Fail to reject H0.")

=== TWO-SAMPLE T-TEST: LENGTH OF STAY ===
Mean Stay (Readmitted <30d): 4.79 days
Mean Stay (Not Readmitted):  4.22 days
T-Statistic: 14.2312
p-value: 2.2944e-45
Cohen's d (Effect Size): 0.1956

Conclusion: Reject H0. Hospital length of stay is significantly longer for patients readmitted within 30 days.


## Hypothesis Test 3: Medication Changes vs Readmission

* **Null Hypothesis ($H_0$):** Changing diabetes medication during the encounter is independent of 30-day hospital readmission.
* **Alternative Hypothesis ($H_1$):** Changing diabetes medication is significantly associated with 30-day hospital readmissions.
* **Test Method:** Chi-Square ($\chi^2$) Test of Independence ($\alpha = 0.05$).

In [9]:
# Create contingency table
contingency_change = pd.crosstab(df['change'], df['readmitted_30'])

# Perform Chi-Square test
chi2_med, p_val_med, dof_med, expected_med = stats.chi2_contingency(contingency_change)

# Calculate Cramér's V
n_med = contingency_change.sum().sum()
min_dim_med = min(contingency_change.shape) - 1
cramers_v_med = np.sqrt(chi2_med / (n_med * min_dim_med))

print("=== CHI-SQUARE TEST: DIABETES MEDICATION CHANGE ===")
print(f"Chi-Square Statistic: {chi2_med:.4f}")
print(f"p-value: {p_val_med:.4e}")
print(f"Cramér's V (Effect Size): {cramers_v_med:.4f}\n")

if p_val_med < 0.05:
    print("Conclusion: Reject H0. Diabetes medication changes show a statistically significant association with early readmission.")
else:
    print("Conclusion: Fail to reject H0.")

=== CHI-SQUARE TEST: DIABETES MEDICATION CHANGE ===
Chi-Square Statistic: 14.6689
p-value: 1.2814e-04
Cramér's V (Effect Size): 0.0145

Conclusion: Reject H0. Diabetes medication changes show a statistically significant association with early readmission.


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Horizontal Bar Plot: P-values summary
plt.figure(figsize=(8, 4))
features = ['Inpatient Visits', 'Num Medications', 'Time in Hospital', 'Num Lab Procedures']
p_values = [0.0001, 0.0001, 0.0001, 0.002]

sns.barplot(
    x=p_values, 
    y=features, 
    hue=features, 
    palette='Blues_r', 
    legend=False
)
plt.axvline(0.05, color='red', linestyle='--', label='Alpha Threshold (0.05)')
plt.title('Hypothesis Testing Significance (p-values)')
plt.xlabel('p-value')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/visualizations/hypothesis_chi2_ttest.png', dpi=300)
plt.show()

<Figure size 800x400 with 1 Axes>

## Preparation Summary

Our statistical hypothesis testing yields three key insights for predictive modeling:

* **Prior Inpatient Admissions:** Showed a highly significant association with 30-day readmissions ($p < 0.001$), confirming prior healthcare utilization as a critical predictive feature.
* **Hospital Length of Stay:** Readmitted patients had a significantly longer mean stay duration ($p < 0.001$), indicating higher clinical acuity correlates with readmission risk.
* **Medication Adjustments:** Changes in diabetes medication were significantly associated with readmissions ($p < 0.001$), pointing to treatment adjustments as a risk signal.

## Next Step
With data cleaning, exploratory statistics, and hypothesis testing complete, the next phase is Feature Engineering and Preprocessing.

Proceed directly to the next notebook:
👉 **[Notebook 04:Feature Engineering and Preprocessing](04_feature_engineering.ipynb).

## Acknowledgements and Credits
Statistical Functions: Implemented using standard routines from SciPy (scipy.stats) and NumPy.

## Sources and Layout Inspiration
**Code Institute Assessment Guidelines:** Structured to meet learning outcomes LO1 and LO3 for formal statistical analysis.

## Tooling and Assistance
**Development Environment:** Visual Studio Code using the Jupyter extension.

**Development & Refinement Notes:**

**Statistical Implementation:** I selected the statistical tests (Chi-Square and Welch's T-Test) and wrote the implementation independently. Gemini assisted with small formula formatting errors when calculating effect sizes (Cramér's V and Cohen's d).

**Text & Flow:** I authored all hypothesis interpretations and summary points directly, using AI as a sounding board to refine sentence flow and ensure professional presentation.